In [0]:
import pandas as pd
from pyspark.sql import SparkSession, Row
import xml.etree.ElementTree as ET
import urllib.request

# Start Spark session
spark = SparkSession.builder.getOrCreate()

# URL of the XML file
url = 'https://raw.githubusercontent.com/Jeffreymaenetja/Code_v2/main/large-dataset.xml'
Bronze_Location = 'dbfs:/FileStore/tables/Bronze/Department_salary'
try:
    # Read XML from URL
    with urllib.request.urlopen(url) as response:
        xml_data = response.read()

    # Parse XML
    root = ET.fromstring(xml_data)

    # Extract <employee> elements
    records = []
    for item in root.findall('.//employee'):
        record = {}
        for element in item:  # <- This line was wrongly indented
            if element.text is not None:
                record[element.tag] = element.text
            else:
                record[element.tag] = ''  # Default value for empty fields
        records.append(Row(**record))  # This must be inside the outer loop

    # Create DataFrame if data exists
    if records:
        df = spark.createDataFrame(records)
        #df.show(5)

        print("Wrirting file to to Bronze location as parquet file ")

        df.write.mode('overwrite').option("header",True).parquet(Bronze_Location)

        print('Writing to file successful!')
    else:
        print("No <employee> elements found in the XML.")

except Exception as e:
    print("Error:", e)


Wrirting file to to Bronze location as parquet file 
